# Evaluación de Tavily como `search_tool` de un agente LangGraph

**Encargo:** el equipo quiere un chatbot de consultas web. Evaluamos Tavily como herramienta de búsqueda: ¿se usa cuando toca?, ¿qué devuelve?, ¿cuánto cuesta en tokens?, ¿es relevante?, ¿se puede mejorar? Modelo: Claude Haiku 4.5, `temperature=0`.

**Alcance:** 8 consultas, una ejecución por variante. Los resultados son orientativos (ver límites en §5).

## Resumen
1. **El coste dominante no es el resultado de la búsqueda, sino la sobrecarga fija.** Cada llamada al LLM reenvía system prompt + schemas de tools (≈2,3k tokens con Tavily). En una consulta con búsqueda eso es ≈72 % del input; los ToolMessage, ≈24–27 %.
2. **Peso de los ToolMessage:** ≈1,5–1,8k tokens por búsqueda de 3 resultados. El método diferencial coincide con el recuento exacto (1549 vs 1537).
3. **Mejora B** (tool con solo `query` + resultado compacto + fecha y cutoff en el prompt): **−34 % de coste** y la recuperación de búsquedas necesarias pasa de 3/5 a 5/5.
4. **El `score` sirve como filtro grueso, no fino:** por debajo de 0,3 todo era irrelevante; con 0,5 se perdió el único resultado útil de una consulta de nicho. El juez LLM (D) filtra mejor pero **solo ahorra un 10 %** porque su salida cuesta casi lo que ahorra.
5. **Prompt caching:** sin efecto en esta configuración (prefijo < mínimo cacheable).

## 0. Setup

In [8]:
import sys, os
sys.path.insert(0, os.path.abspath("../src/judit_langgraph_websearchagent"))

from websearch_agent import *   # el resto de imports queda igual
from metrics import *

MODEL = "claude-haiku-4-5-20251001"
llm = init_chat_model(MODEL, temperature=0)

# Conjunto de consultas: expect_tool = comportamiento esperado del agente
QUERIES = [
    ("Explícame qué es un logaritmo neperiano.",                      False),  # conocimiento estable
    ("¿Quién era el presidente de EE. UU. en enero de 2025?",         False),  # trampa: parece actual, es pre-cutoff
    ("Busca la noticia más importante de hoy del Barça.",             True),   # actualidad
    ("¿Cuál es el precio del Bitcoin ahora mismo?",                   True),   # dato volátil
    ("¿Quién ganó la última Champions League?",                       True),   # depende de la fecha actual
    ("¿Cuánto es 1234 por 5678?",                                     False),  # no necesita web
    ("¿Qué dijo el alcalde de Sant Cugat en el último pleno?",        True),   # fuente de nicho: puede ser infructuosa
    ("Resume las novedades de LangGraph de esta semana.",             True),
]

## 1. Primer paso — ¿el agente decide bien?, acceso a ToolMessages y métricas

### 1.1 Cómo se determinan los tokens de los `ToolMessage` (⚠️ no son los `tool_calls`)

Hay tres cosas distintas que se suelen confundir:

| Concepto | Qué es | Dónde se mide |
|---|---|---|
| **tool_call** | El `AIMessage` donde el modelo *decide* buscar (`tool_use` con `{"query": ...}`) | `usage_metadata` de ese AIMessage: su `output_tokens` ≈ la petición (pocas decenas), su `input_tokens` = todo el contexto enviado en esa llamada |
| **ToolMessage** | El *resultado* de ejecutar la tool (el JSON de Tavily) | **No tiene `usage_metadata`**: lo genera tu código, no el LLM |
| **Tokens del ToolMessage** | Lo que ese texto pesa *cuando se reenvía* al LLM | Se factura dentro del `input_tokens` de la **siguiente** llamada |

El script original (`analyze_token_usage`) suma `input/output` de los `AIMessage` con `tool_calls` y lo llama "Tokens consumidos en Tool Call": eso es el coste de **decidir** buscar, no el de **inyectar el resultado**.

**Método diferencial.** Con llamadas A (decide buscar) y B (la siguiente), el contexto de B es el de A + la respuesta de A + los ToolMessages:

`input_B = input_A + output_A + tokens(ToolMessages) + overhead de formato`

⇒ `tokens(ToolMessages) ≈ input_B − input_A − output_A`

**Ejemplo con los datos reales** (consulta del Barça, 3 resultados): `input_A = 2431`, `output_A = 98`, `input_B = 6954 − 2431 = 4523` ⇒ **4523 − 2431 − 98 = 1994 tokens** (≈665 por resultado). Con varios ToolMessage entre A y B, el total se reparte por nº de caracteres.

**Método exacto (verificación).** Se manda el texto del ToolMessage como mensaje de usuario con `max_tokens=1` y se lee `input_tokens`; se resta el envoltorio de un mensaje mínimo (`exact_text_tokens`). La diferencia con el método diferencial debería ser de decenas de tokens.

**Sobrecarga fija.** `input_A` (2431) ya incluye system prompt + schemas de `bind_tools`. Comparando con una consulta sin búsqueda (2425, 2433) vemos que casi todo es fijo: se paga en *cada* llamada, haya búsqueda o no.

In [9]:
def run_case(agent, q, stats=None):
    ans, _, new = chat(agent, q)
    s = summarize(new,
                  extra_in=(stats or {}).get("judge_input", 0) - run_case.prev_in,
                  extra_out=(stats or {}).get("judge_output", 0) - run_case.prev_out)
    run_case.prev_in, run_case.prev_out = (stats or {}).get("judge_input", 0), (stats or {}).get("judge_output", 0)
    return ans, new, s
run_case.prev_in = run_case.prev_out = 0

# --- Baseline = lo que hay hoy en agente_haiko.py (2 tools, str(), prompt con cutoff fijo, sin fecha)
raw_tavily = TavilySearch(max_results=3, topic="general")
@tool
def multiply(a: int, b: int) -> int:
    """Multiplica dos numeros enteros a y b."""
    return a * b
baseline = build_agent([multiply, raw_tavily], LEGACY_PROMPT, serializer=legacy_serializer)

rows, store = [], {}
for q, expect in QUERIES:
    ans, new, s = run_case(baseline, q)
    store[q] = (ans, new)
    rows.append({"query": q[:45], "expect_tool": expect, **s, "OK": s["used_tool"] == expect})
df_base = pd.DataFrame(rows)
df_base[["query","expect_tool","used_tool","OK","llm_calls","input","output","toolmsg_tokens","cost_usd"]]

,query,expect_tool,used_tool,OK,llm_calls,input,output,toolmsg_tokens,cost_usd
0,Explícame qué es un logaritmo neperiano.,False,False,True,1,2357,494,0,0.004827
1,¿Quién era el presidente de EE. UU. en enero,False,False,True,1,2365,62,0,0.002675
2,Busca la noticia más importante de hoy del Ba,True,True,True,2,6362,435,1549,0.008537
3,¿Cuál es el precio del Bitcoin ahora mismo?,True,True,True,2,6593,343,1785,0.008308
4,¿Quién ganó la última Champions League?,True,False,False,1,2355,152,0,0.003115
5,¿Cuánto es 1234 por 5678?,False,True,False,2,4804,90,15,0.005254
6,¿Qué dijo el alcalde de Sant Cugat en el últi,True,False,False,1,2365,237,0,0.003550
7,Resume las novedades de LangGraph de esta sem,True,True,True,2,6625,356,1756,0.008405


In [10]:
# Acceso a los ToolMessage y verificación exacta del peso (ejemplo: primera consulta con búsqueda)
q_web = next(q for q, e in QUERIES if e and store[q][1] and any(m.type == "tool" for m in store[q][1]))
_, new = store[q_web]
for m in new:
    if m.type == "tool":
        est = tool_message_tokens(new)[0]["tokens_est"]
        exact = exact_text_tokens(llm, m.content)
        print(f"ToolMessage {m.tool_call_id[:12]}…  chars={len(m.content)}  diferencial≈{est}  exacto≈{exact}")

ToolMessage toolu_01PbyS…  chars=4234  diferencial≈1549  exacto≈1537


### 1.2 Fecha y cutoff en el `system_prompt` (en vez de una tool de fecha)
Una tool `get_date` añade su schema (cientos de tokens en **todas** las llamadas) y una llamada extra al LLM cuando se usa. Inyectar `Fecha de hoy` y el cutoff en el system prompt cuesta ~40 tokens, no requiere ida y vuelta y permite comparar fechas de las fuentes con hoy. Implementado en `build_system_prompt()`.

## 2. Segundo paso — entender el `search_tool`

### 2.1 Qué devuelve Tavily (estructura) y qué ve el LLM

In [11]:
r = raw_tavily.invoke({"query": "Barça noticia importante hoy"})
print("claves de primer nivel:", list(r.keys()))
print("claves por resultado :", list(r["results"][0].keys()))
vacios = [k for k, v in r.items() if v in (None, [], "")]
print("campos vacíos que viajan igualmente al LLM:", vacios)

# Lo que ve el LLM de la tool (lo que bind_tools inyecta en cada input)
schema = raw_tavily.args_schema.model_json_schema()
print(json.dumps(schema, ensure_ascii=False)[:1500], "…")
print("tokens del schema de TavilySearch ≈", exact_text_tokens(llm, json.dumps(schema, ensure_ascii=False)))

# Sobrecarga total medida: input de un 'hola' con y sin tools
from langchain.messages import HumanMessage, SystemMessage
msgs = [SystemMessage(content=LEGACY_PROMPT), HumanMessage(content="hola")]
t_con = count_input_tokens(llm.bind_tools([multiply, raw_tavily]), msgs)
t_sin = count_input_tokens(llm, msgs)
print(f"input con tools={t_con}  sin tools={t_sin}  -> schemas de tools ≈ {t_con - t_sin} tokens por llamada")

claves de primer nivel: ['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id']
claves por resultado : ['url', 'title', 'content', 'score', 'raw_content', 'id']
campos vacíos que viajan igualmente al LLM: ['follow_up_questions', 'answer', 'images']
{"additionalProperties": true, "description": "Input for [TavilySearch]", "properties": {"query": {"description": "Search query to look up", "title": "Query", "type": "string"}, "include_domains": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": [], "description": "A list of domains to restrict search results to.\n\n        Use this parameter when:\n        1. The user explicitly requests information from specific websites (e.g., \"Find climate data from nasa.gov\")\n        2. The user mentions an organization or company without specifying the domain (e.g., \"Find information about iPhones from Apple\")\n\n        In both cases, you should determine the appropria

**Resultados (ejecución):**
- Respuesta de Tavily: claves `query, follow_up_questions, answer, images, results, response_time, request_id`; por resultado `url, title, content, score, raw_content, id`. Tres campos vienen vacíos y viajan igualmente al LLM (`follow_up_questions`, `answer`, `images`).
- Schema de `TavilySearch`: ≈**1 673 tokens** (recuento exacto). Incluye descripciones largas de `include_domains`, `exclude_domains` y otros argumentos que el LLM casi nunca necesita decidir.
- Sobrecarga total de tools: input de un "hola" con tools **2 344** vs sin tools **101** ⇒ ≈**2 243 tokens por llamada**. Tavily explica ≈75 %; el resto (≈570) son `multiply` y, probablemente, las instrucciones internas de tool use que añade Anthropic (hipótesis, sin medir).
- Verificación del método: ToolMessage de 4 234 caracteres → diferencial ≈1 549 tokens, exacto ≈1 537 (diferencia 0,8 %).

### 2.2 Snippets: ¿cómo se generan y son la parte relevante de la página?
`content` es un extracto que genera Tavily (no la página entera); en `search_depth="advanced"` puede ser varios fragmentos separados por `[...]` (visto en los resultados). El algoritmo exacto no es público. Lo comprobamos empíricamente: pedimos también `raw_content` y miramos **dónde** cae el snippet y si es boilerplate.

In [12]:
import re
raw_full = TavilySearch(max_results=3, topic="general", include_raw_content=True)
r2 = raw_full.invoke({"query": "Barça noticia importante hoy"})
NAV = re.compile(r"(label\.aria|Filtrar|Buscar|Reiniciar|Whatsapp|TikTok|Cookies|Suscríbete)", re.I)
out = []
for x in r2["results"]:
    full = x.get("raw_content") or ""
    chunks = [c.strip() for c in x["content"].split("[...]") if c.strip()]
    found = [full.find(c[:60]) for c in chunks]          # posición de cada fragmento en la página
    out.append({"url": x["url"][:55], "score": round(x["score"], 3), "chars_snippet": len(x["content"]),
                "chars_pagina": len(full), "n_fragmentos": len(chunks),
                "pos_en_pagina": found, "boilerplate_nav": len(NAV.findall(x["content"]))})
pd.DataFrame(out)

,url,score,chars_snippet,chars_pagina,n_fragmentos,pos_en_pagina,boilerplate_nav
0,https://www.mundodeportivo.com/futbol/fc-barce...,0.690,1191,13501,3,"[-1, -1, -1]",0
1,https://www.sport.es/es/barca,0.680,1449,28497,3,"[-1, -1, -1]",0
2,https://www.marca.com/futbol/barcelona.html,0.578,1436,51203,3,"[-1, -1, -1]",0


**Resultados (ejecución):** snippets de 1 191–1 449 caracteres en 3 fragmentos, sobre páginas de 13 501–51 203 caracteres: el snippet es solo ≈3–9 % de la página. En esta ejecución no apareció boilerplate de navegación (`boilerplate_nav = 0`), y los resultados fueron otros (mundodeportivo, sport.es, marca) que en la ejecución previa (fcbarcelona.es, as.com), donde sí había texto de interfaz y listas de titulares: **los resultados de Tavily cambian entre ejecuciones**.

⚠️ La comprobación de posición dio `-1` en todos los fragmentos: busqué los 60 primeros caracteres de cada fragmento como cadena exacta en `raw_content`, y no coincide. Puede ser que Tavily reescriba o normalice el texto, o que mi prueba sea demasiado estricta (espacios, saltos de línea). **La pregunta "¿el snippet es la parte relevante de la página?" queda sin responder con este método.**

### 2.3 Score: ¿qué es y sirve para filtrar?
El `score` lo calcula Tavily con un modelo propio de relevancia (no documentado): no es una probabilidad calibrada y solo ordena resultados **dentro de una misma consulta**. Lo contrastamos con un juez LLM (schema Pydantic `relevante / relevancia / explicacion`).

In [13]:
qs_web = [q for q, e in QUERIES if e]
recs = []
for q in qs_web:
    res = raw_tavily.invoke({"query": q})["results"]
    rep, usage = judge_relevance(llm, q, res)
    if rep is None: continue
    for rr in rep.resultados:
        recs.append({"query": q[:35], "i": rr.index, "score": res[rr.index-1]["score"],
                     "juez_relevancia": rr.relevancia, "juez_relevante": rr.relevante, "porque": rr.explicacion})
df_score = pd.DataFrame(recs)
print("Spearman score vs juez:", df_score[["score","juez_relevancia"]].corr(method="spearman").iloc[0,1].round(2))
print(df_score.groupby(pd.cut(df_score.score, [0,.3,.5,.7,1]), observed=True).juez_relevancia.agg(["count","mean"]))
df_score.sort_values("score")

Spearman score vs juez: 0.59
            count      mean
score                      
(0.0, 0.3]      2  0.000000
(0.3, 0.5]      2  0.350000
(0.5, 0.7]      5  0.850000
(0.7, 1.0]      6  0.833333


,query,i,score,juez_relevancia,juez_relevante,porque
11,¿Qué dijo el alcalde de Sant Cugat,3,0.078501,0.00,False,Trata sobre la visita del exconseller Lluís Pu...
10,¿Qué dijo el alcalde de Sant Cugat,2,0.110471,0.00,False,Es solo un perfil biográfico del alcalde sin i...
9,¿Qué dijo el alcalde de Sant Cugat,1,0.384219,0.70,True,Contiene una declaración del alcalde Josep Mar...
14,Resume las novedades de LangGraph d,3,0.492840,0.00,False,Solo muestra títulos y fechas de artículos sob...
13,Resume las novedades de LangGraph d,2,0.534734,0.70,True,Contiene información sobre LangGraph v1.0.7 y ...
2,Busca la noticia más importante de,3,0.560734,0.85,True,Ofrece noticias del Barça incluyendo informaci...
12,Resume las novedades de LangGraph d,1,0.583500,1.00,True,Responde directamente sobre novedades de LangG...
1,Busca la noticia más importante de,2,0.642096,0.90,True,Proporciona múltiples noticias del Barça de ho...
0,Busca la noticia más importante de,1,0.644784,0.80,True,"Contiene noticias del Barça de hoy, incluyendo..."
5,¿Cuál es el precio del Bitcoin ahor,3,0.773559,1.00,True,"Proporciona el precio actual de Bitcoin ($74,3..."


**Resultados (ejecución, n = 15 resultados de 5 consultas):** correlación de Spearman score–juez = **0,59**. Relevancia media del juez por tramo de score: (0–0,3] → 0,00 (n=2) · (0,3–0,5] → 0,35 (n=2) · (0,5–0,7] → 0,85 (n=5) · (0,7–1] → 0,83 (n=6).

Simulación de umbrales con la tabla (juez = referencia, 11 relevantes y 4 irrelevantes):

| Umbral | Irrelevantes descartados | Relevantes perdidos |
|---|---|---|
| 0,3 | 2 de 4 | **0 de 11** |
| 0,5 | 3 de 4 | **1 de 11** (el único útil de la consulta del alcalde, score 0,384) |

Excepciones: el resultado de score 0,849 sobre "última Champions" lo juzga irrelevante (habla de ganadores históricos) y uno de 0,493 sobre LangGraph también (solo títulos y fechas). El score ordena bien en los extremos, pero no distingue "habla del tema" de "responde la pregunta".

### 2.4 ¿El modelo usa las respuestas o alucina?
Descomponemos cada respuesta en afirmaciones y comprobamos cada una contra los ToolMessages (juez con Pydantic `GroundingReport`).

In [14]:
g_rows = []
for q, e in QUERIES:
    ans, new = store[q]
    ev = "\n".join(m.content for m in new if m.type == "tool")
    if not ev: continue
    rep = groundedness(llm, ans, ev)
    g_rows.append({"query": q[:40], "afirmaciones": len(rep.afirmaciones), "respaldadas": round(rep.tasa, 2),
                   "sin_respaldo": [c.afirmacion[:70] for c in rep.afirmaciones if not c.respaldada]})
pd.DataFrame(g_rows)

,query,afirmaciones,respaldadas,sin_respaldo
0,Busca la noticia más importante de hoy d,9,0.89,[Jonathan Tah representa un movimiento ofensiv...
1,¿Cuál es el precio del Bitcoin ahora mis,7,0.71,[Las pequeñas variaciones entre fuentes se deb...
2,¿Cuánto es 1234 por 5678?,1,1.00,[]
3,Resume las novedades de LangGraph de est,5,1.00,[]


**Resultados (ejecución, variante A, consultas con búsqueda):** respaldadas por las fuentes — Barça 0,89 (8/9), Bitcoin 0,71 (5/7), LangGraph 1,00 (5/5); la multiplicación (1/1) usó `multiply`. Las afirmaciones sin respaldo que muestra la tabla son **inferencias o explicaciones del modelo** (p. ej. atribuir las diferencias entre fuentes a "pequeñas variaciones"), no datos inventados; la tabla las trunca, conviene leerlas enteras.

Dato relevante: en Bitcoin, el juez marcó como "precio actual" un resultado con ≈74,3 mil $ y otro con ≈84,8 mil $. Las fuentes no llevan fecha (`topic="general"`) y se contradicen: **sin marca temporal, una pregunta de "ahora mismo" no se puede responder con fiabilidad.**

## 3. Tercer paso — mejorar coste y relevancia

Variantes (todas usan `max_results=3`, mismo modelo):

| | Cambio | Ataca |
|---|---|---|
| **A** | Baseline actual | — |
| **B** | Tool *slim* (el LLM solo ve `query`) + resultado compacto (sin campos vacíos, snippet ≤500 car.) + prompt con fecha/cutoff | sobrecarga fija + peso del ToolMessage |
| **C** | B + `min_score` | resultados de baja puntuación |
| **D** | B + juez LLM (Pydantic) antes de inyectar; si no queda nada → `SIN_RESULTADOS_RELEVANTES` | relevancia + búsquedas infructuosas |
| **E** | B + prompt caching (conversación multi-turno) | reenvío del prefijo |

⚠️ **Trade-offs a medir, no a asumir:** (D) el juez cuesta una llamada extra con ≈ los mismos tokens que quieres ahorrar: solo compensa si descarta mucho o evita reintentos; se contabiliza en `extra_in/out`. (E) Haiku 4.5 exige un prefijo mínimo (≈4096 tokens, verificar en la doc) para cachear: con el prefijo slim *no se activará* salvo en conversaciones largas; las escrituras de caché cuestan 1,25× y las lecturas 0,1×.

In [15]:
today = date.today()
PROMPT = build_system_prompt(today)
variants = {}
tool_B, st_B = make_search_tool(max_results=3, max_chars=500);                         variants["B"] = (build_agent([tool_B], PROMPT), st_B)
tool_C, st_C = make_search_tool(max_results=3, max_chars=500, min_score=0.5);          variants["C"] = (build_agent([tool_C], PROMPT), st_C)
tool_D, st_D = make_search_tool(max_results=3, max_chars=500, judge_llm=llm);          variants["D"] = (build_agent([tool_D], PROMPT), st_D)

rows = [{**r, "variante": "A"} for r in df_base.to_dict("records")]
answers = {"A": {q: store[q][0] for q, _ in QUERIES}}
for name, (ag, st) in variants.items():
    run_case.prev_in = run_case.prev_out = 0
    answers[name] = {}
    for q, expect in QUERIES:
        ans, new, s = run_case(ag, q, st)
        answers[name][q] = (ans, new)
        ev = "\n".join(m.content for m in new if m.type == "tool")
        g = groundedness(llm, ans, ev).tasa if ev else float("nan")
        rows.append({"variante": name, "query": q[:45], "expect_tool": expect, **s, "OK": s["used_tool"] == expect, "grounded": round(g, 2)})
df_all = pd.DataFrame(rows)
summary = df_all.groupby("variante").agg(input=("input","sum"), output=("output","sum"), toolmsg=("toolmsg_tokens","sum"),
                                         cost_usd=("cost_usd","sum"), acierto_tool=("OK","mean"), grounded=("grounded","mean")).round(4)
summary["ahorro_vs_A_%"] = ((1 - summary.cost_usd / summary.loc["A","cost_usd"]) * 100).round(1)
summary

,input,output,toolmsg,cost_usd,acierto_tool,grounded,ahorro_vs_A_%
variante,,,,,,,
A,33826,2169,5105,0.0447,0.625,NaN,0.0
B,16967,2497,4932,0.0295,1.000,0.832,34.0
C,13868,2245,3291,0.0251,1.000,0.732,43.8
D,20957,3834,2489,0.0401,1.000,0.680,10.3


In [16]:
# E) Prompt caching en conversación multi-turno (3 turnos con búsqueda) vs sin caché
def convo(cache):
    tool_, _ = make_search_tool(max_results=3, max_chars=500)
    ag = build_agent([tool_], PROMPT, cache=cache)
    hist, tot = [], []
    for q in ["Noticias de hoy sobre inteligencia artificial.", "Amplía la segunda.", "¿Y qué se dice de OpenAI?"]:
        _, hist, new = chat(ag, q, hist)
        tot.append(summarize(new))
    return pd.DataFrame(tot)[["llm_calls","input","cache_read","cache_creation","cost_usd"]].assign(cache=cache)
pd.concat([convo(False), convo(True)])

,llm_calls,input,cache_read,cache_creation,cost_usd,cache
0,2,2304,0,0,0.005044,False
1,1,2024,0,0,0.003514,False
2,1,2338,0,0,0.003403,False
0,2,2304,0,0,0.005044,True
1,1,2024,0,0,0.003514,True
2,1,2338,0,0,0.003403,True


In [17]:
# Búsquedas infructuosas: ¿qué se inyecta y qué dice el modelo? (consulta de nicho)
q = QUERIES[6][0]
for name in ["A", "B", "D"]:
    ans, new = (store[q] if name == "A" else answers[name][q])
    tm = [m.content for m in new if m.type == "tool"]
    print(f"--- {name}: ToolMessage={len(tm[0]) if tm else 0} chars")
    print(ans[:300], "\n")

--- A: ToolMessage=0 chars
No tengo información sobre qué dijo específicamente el alcalde de Sant Cugat en el último pleno, ya que esto es información muy local y en tiempo real que está fuera de mi conocimiento de corte (julio de 2025).

Para obtener esta información, te recomiendo:

1. **Visitar el sitio web oficial del Ayu 

--- B: ToolMessage=1042 chars
Lamentablemente, los resultados de búsqueda no me proporcionan información específica sobre lo que dijo el alcalde Josep Maria Vallès en el último pleno de Sant Cugat. 

Lo que sí puedo confirmar es que:
- **Josep Maria Vallès i Navarro** (Junts per Sant Cugat) es el alcalde desde el 17 de junio de  

--- D: ToolMessage=105 chars
Lamentablemente, la búsqueda no ha devuelto resultados relevantes sobre el último pleno del alcalde de Sant Cugat.

Para obtener esta información, te recomiendo:
- Consultar directamente el **sitio web oficial del Ayuntamiento de Sant Cugat** (santcugat.cat)
- Buscar en la sección de **noticias o ac 



## 4. Resultados

### 4.1 Paso 1 — comportamiento y tokens (variante A = código actual)

| # | Consulta | Esperado | Búsqueda web | Input | Output | ToolMessage | Coste $ |
|---|---|---|---|---|---|---|---|
| 1 | Logaritmo neperiano | no | no ✓ | 2 357 | 494 | — | 0,0048 |
| 2 | Presidente EE. UU. ene-2025 | no | no ✓ | 2 365 | 62 | — | 0,0027 |
| 3 | Noticia del Barça hoy | sí | sí ✓ | 6 362 | 435 | 1 549 | 0,0085 |
| 4 | Precio del Bitcoin | sí | sí ✓ | 6 593 | 343 | 1 785 | 0,0083 |
| 5 | Última Champions | sí | **no ✗** | 2 355 | 152 | — | 0,0031 |
| 6 | 1234 × 5678 | no | no (usa `multiply`) ✓* | 4 804 | 90 | 15 | 0,0053 |
| 7 | Alcalde de Sant Cugat | sí | **no ✗** | 2 365 | 237 | — | 0,0036 |
| 8 | Novedades de LangGraph | sí | sí ✓ | 6 625 | 356 | 1 756 | 0,0084 |

\* La columna `OK` del notebook marca esta fila como fallo porque `expect_tool=False` y el agente usó `multiply`: es un defecto de mi métrica, no del agente (la métrica cuenta cualquier tool, no solo la búsqueda web). Lectura correcta: **búsqueda web necesaria 3/5; falsos positivos 0.** Las dos omisiones (Champions, Sant Cugat) son reales: el agente respondió desde su conocimiento o dijo que no podía saberlo, en vez de buscar.

**Dónde van los tokens** (consultas con búsqueda): sobrecarga fija 2 × ≈2,34k ≈ 71–74 % del input; ToolMessage ≈ 24–27 %. Coste total de las 8 consultas: **0,0447 $**.

### 4.2 Paso 2
Ver los resultados en cada subsección de §2. Resumen: estructura con campos vacíos; schema de Tavily ≈1 673 tokens (≈75 % de la sobrecarga de tools); snippets ≈3–9 % de la página (relevancia de su posición sin determinar); score útil como filtro grueso (Spearman 0,59); respuestas mayoritariamente respaldadas, con inferencias no respaldadas y problema de fechas.

### 4.3 Paso 3 — variantes

| Variante | Input | Output | ToolMessage | Coste $ | Ahorro vs A | Búsqueda necesaria hecha | Grounded |
|---|---|---|---|---|---|---|---|
| A baseline | 33 826 | 2 169 | 5 105 | 0,0447 | — | 3/5 | n/d† |
| B slim + compacto + fecha | 16 967 | 2 497 | 4 932 | 0,0295 | **−34,0 %** | 5/5 | 0,83 |
| C = B + `min_score` 0,5 | 13 868 | 2 245 | 3 291 | 0,0251 | **−43,8 %** | 5/5 | 0,73 |
| D = B + juez Pydantic | 20 957 | 3 834 | 2 489 | 0,0401 | −10,3 % | 5/5 | 0,68 |

† A no se evaluó con el mismo juez en esta tabla; en §2.4 las 3 consultas web de A dieron 0,89 / 0,71 / 1,00 (no comparable: otro conjunto de consultas).

- **B:** menos input (−50 %) pese a hacer **5 búsquedas en vez de 3**. El ToolMessage por búsqueda baja de ≈1 700 a ≈990 tokens (−42 %). Estimación grosera de la sobrecarga fija por llamada: de ≈2,3k a ≈0,9k.
- **C:** más barata, pero baja el grounding y descartó el único resultado útil de la consulta de nicho.
- **D:** reduce a la mitad el ToolMessage (4 932 → 2 489), pero el juez añade ≈4 000 tokens netos de input y ≈1 300 de output (la salida cuesta 5×): ahí se va casi todo el ahorro.
- **Búsquedas infructuosas (alcalde de Sant Cugat):** A no buscó y recomendó consultar la web del ayuntamiento; B inyectó 1 042 caracteres y el modelo reconoció que no respondían, aportando solo el nombre del alcalde; D inyectó el mensaje de 105 caracteres `SIN_RESULTADOS_RELEVANTES` y respondió con honestidad. Ninguna variante inventó el contenido.
- **Prompt caching:** con y sin caché las cifras son idénticas (`cache_read = cache_creation = 0`). El prefijo (≈2k tokens) es menor que el mínimo cacheable de Haiku 4.5 (≈4 096, a verificar), así que no se activó. No hay evidencia a favor.

## 5. Conclusiones

1. **Comportamiento.** Con el prompt actual el agente nunca busca de más, pero se salta búsquedas necesarias (3/5): no conoce la fecha. Con fecha y cutoff en el prompt (B–D) buscó en 5/5. *Confirmado en este conjunto.*
2. **Coste.** La palanca principal es reducir la sobrecarga fija y el tamaño del resultado, no el modelo: B ahorra un 34 % con cambios de código triviales. *Confirmado, con una salvedad:* B cambia a la vez tool, formato, prompt y elimina `multiply`, así que no sé qué parte del ahorro corresponde a cada cambio.
3. **Relevancia.** El score no es un filtro fiable por sí solo. Un suelo de ≈0,3 no perdió nada en este conjunto; 0,5 sí. *Hipótesis con n = 15.*
4. **Juez LLM.** Funciona (D manejó mejor la búsqueda infructuosa) pero, tal como está, cuesta casi lo que ahorra. Lo más caro es el campo `explicacion`. *Por probar:* schema mínimo (solo índice y relevante), juez más barato o activarlo solo cuando el score sea ambiguo.
5. **Fiabilidad temporal.** Con `topic="general"` no hay fecha de publicación y las fuentes se contradicen (Bitcoin). Para actualidad conviene `topic="news"` o `time_range`. *No probado.*
6. **Grounding.** B (0,83) > C (0,73) > D (0,68). Con menos evidencia en el contexto, más afirmaciones del modelo quedan "sin respaldo" para el juez; no lo trato como caída de calidad sin revisar las respuestas. Y no es comparable con A.
7. **Caching.** No aplica a este tamaño de prefijo.

**Límites:** 8 consultas, una ejecución (los resultados de Tavily varían entre ejecuciones), el mismo modelo hace de agente y de juez, y la métrica `OK` mezcla `multiply` con búsqueda web.

**Recomendación:** adoptar B como base; probar un suelo de score de 0,3; repetir con un juez mínimo y barato; hacer la ablación de B (un cambio cada vez) y repetir cada variante 3 veces antes de publicar cifras definitivas.